下の枠にマウスを移動すると表示される、直下の枠左上の三角印をクリックしてください。動作してグラフが表示される場合まで15秒ほどかかります。

In [ ]:
!pip install ipywidgets
from ipywidgets import interact, FloatSlider, ToggleButtons
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# 物性値（温度 22.5℃）
# -----------------------------
beta = 3.0e-4
nu   = 0.9e-6
alpha = 0.14e-6
k = 0.6
g = 9.81

# -----------------------------
# Rayleigh 数
# -----------------------------
def calc_Ra(H, dT):
    return g * beta * dT * H**3 / (nu * alpha)

# -----------------------------
# Nusselt 数
# -----------------------------
def calc_Nu(Ra, C):
    return C * Ra**(1/3)

# -----------------------------
# 熱貫流率 U
# -----------------------------
def calc_U(Nu, H):
    return Nu * k / H

# -----------------------------
# 流れ場の可視化
# -----------------------------
def visualize_field(mode):
    nx, ny = 40, 40
    x = np.linspace(0, 1, nx)
    y = np.linspace(0, 1, ny)
    X, Y = np.meshgrid(x, y)

    plt.figure(figsize=(6,5))

    if mode == "Bottom heating (convection ON)":
        A = 1.0
        u =  A * np.sin(np.pi * X) * np.cos(np.pi * Y)
        v = -A * np.cos(np.pi * X) * np.sin(np.pi * Y)

        speed = np.sqrt(u**2 + v**2)

        plt.streamplot(X, Y, u, v, color=speed, cmap='turbo')
        plt.title("Convection streamlines (Bottom heating)")
        plt.colorbar(label="Velocity magnitude")

    else:
        T = 1 - Y
        plt.contourf(X, Y, T, levels=20, cmap='coolwarm')
        plt.title("Isotherms (Top heating)")
        plt.colorbar(label="Temperature")

    plt.gca().invert_yaxis()
    plt.xlabel("x")
    plt.ylabel("y")
    plt.show()

# -----------------------------
# インタラクティブモデル
# -----------------------------
def interactive_model(H_mm, dT, C, mode):
    H = H_mm / 1000

    if mode == "Bottom heating (convection ON)":
        Ra = calc_Ra(H, dT)
        Nu = calc_Nu(Ra, C)
    else:
        Ra = calc_Ra(H, dT)
        Nu = 1.0

    U = calc_U(Nu, H)

    print(f"Mode = {mode}")
    print(f"Rayleigh number Ra = {Ra:.3e}")
    print(f"Nusselt number Nu = {Nu:.2f}")
    print(f"U-value = {U:.1f} W/(m2·K)")

    # 厚さと U の関係
    H_list = np.linspace(0.03, 0.12, 50)
    U_list = []

    for H_i in H_list:
        Ra_i = calc_Ra(H_i, dT)
        Nu_i = calc_Nu(Ra_i, C) if mode == "Bottom heating (convection ON)" else 1.0
        U_list.append(calc_U(Nu_i, H_i))

    plt.figure(figsize=(7,5))
    plt.plot(H_list*1000, U_list, lw=2)
    plt.axvline(H_mm, color='red', linestyle='--')
    plt.xlabel("Thickness H [mm]")
    plt.ylabel("U-value [W/(m2·K)]")
    plt.title("Effect of water layer thickness on U-value")
    plt.grid(True)
    plt.show()

    visualize_field(mode)

# -----------------------------
# UI
# -----------------------------
interact(
    interactive_model,
    H_mm = FloatSlider(value=90, min=30, max=120, step=1, description="H [mm]"),
    dT   = FloatSlider(value=15, min=5, max=25, step=1, description="ΔT [K]"),
    C    = FloatSlider(value=0.069, min=0.03, max=0.12, step=0.001, description="C coeff"),
    mode = ToggleButtons(
        options=["Bottom heating (convection ON)", "Top heating (convection OFF)"],
        description="Heating mode"
    )
);
